# MC-OCR 2021: Dataset Audit & Evaluation Preparation
### Unsupervised Receipt Understanding

**Execution Environment:** Kaggle Notebook (CPU/GPU)
**Dataset Input:** `domixi1989/vietnamese-receipts-mc-ocr-2021` (attach via **Add Input**)

**Purpose:**
This notebook is for dataset auditing and later final evaluation only. Ground-truth entity labels must not be used to train or tune the unsupervised inference pipeline.

**Objectives:**
1. Dynamically inspect `/kaggle/input` and discover actual dataset folders.
2. Identify image directories and annotation CSV safely without hardcoded paths.
3. Parse annotations (`anno_polygons`, `anno_texts`, `anno_labels`) for data-quality auditing only.
4. Validate structural alignment between bounding polygons, text tokens, and labels.
5. Detect missing values, empty records, and label anomalies (e.g. `TOTAL_TOTAL_COST`).
6. Inspect image readability, resolution distributions, and aspect ratios.
7. Optionally visualize sample annotations for audit; do not feed them into inference or model fitting.
8. Export a machine-readable data quality report (`mc_ocr_data_quality_report.json`).

In [ ]:
import os
import sys
import json
import ast
from pathlib import Path
from typing import Dict, List, Any

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Matplotlib visual settings
plt.rcParams['figure.dpi'] = 100

print(f'Python version: {sys.version}')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')

In [ ]:
# Dynamic input discovery: checks Kaggle /kaggle/input, falls back to local data/raw
def discover_data_root() -> Path:
    kaggle_base = Path('/kaggle/input')
    if kaggle_base.exists():
        candidates = list(kaggle_base.glob('*mc-ocr*')) + list(kaggle_base.glob('*vietnamese*'))
        if candidates:
            return candidates[0]
        subdirs = [p for p in kaggle_base.iterdir() if p.is_dir()]
        if subdirs:
            return subdirs[0]
    local_candidates = [Path('data/raw/mc-ocr-2021'), Path('data/raw'), Path('../data/raw'), Path('../../data/raw')]
    for p in local_candidates:
        if p.exists() and (p / 'mcocr_train_df.csv').exists():
            return p
    raise FileNotFoundError('Could not locate dataset root in /kaggle/input or local data/raw!')
DATA_ROOT = discover_data_root()
print(f'==> Discovered DATA_ROOT: {DATA_ROOT.resolve()}')
print('\nDiscovered files and directories:')
for item in sorted(DATA_ROOT.iterdir()):
    if item.is_dir():
        sub_count = len(list(item.iterdir()))
        print(f'  📁 {item.name}/ ({sub_count} items)')
    else:
        size_mb = item.stat().st_size / (1024 * 1024)
        print(f'  📄 {item.name} ({size_mb:.2f} MB)')


In [ ]:
# Locate image directory dynamically (handles nested unzips like train_images/train_images/)
def find_image_dir(root: Path, target_name: str = 'train_images') -> Path:
    direct = root / target_name
    if direct.exists():
        nested = direct / target_name
        if nested.exists() and any(nested.glob('*.jpg')):
            return nested
        if any(direct.glob('*.jpg')):
            return direct
    matches = [p for p in root.rglob(target_name) if p.is_dir() and any(p.glob('*.jpg'))]
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Could not locate image directory for {target_name} under {root}')

TRAIN_CSV_PATH = DATA_ROOT / 'mcocr_train_df.csv'
TRAIN_IMG_DIR = find_image_dir(DATA_ROOT, 'train_images')

print(f'Train CSV Path: {TRAIN_CSV_PATH}')
print(f'Train Image Dir: {TRAIN_IMG_DIR} ({len(list(TRAIN_IMG_DIR.glob("*.jpg")))} images found)')

In [ ]:
# Load CSV and inspect structure without schema assumptions
df_raw = pd.read_csv(TRAIN_CSV_PATH)
print(f'Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns')
print('\nColumns & Data Types:')
for col, dtype in df_raw.dtypes.items():
    print(f'  - {col}: {dtype}')

print('\nMissing Values Count:')
for col, missing in df_raw.isnull().sum().items():
    print(f'  - {col}: {missing}')

df_raw.head(3)

In [ ]:
# Label taxonomy and anomaly normalization
LABEL_NORMALIZATION = {
    'SELLER': 'SELLER',
    'ADDRESS': 'ADDRESS',
    'TIMESTAMP': 'TIMESTAMP',
    'TOTAL_COST': 'TOTAL_COST',
    'TOTAL_TOTAL_COST': 'TOTAL_COST'  # Fix anomaly discovered in row 8
}

def parse_polygons_field(poly_val: Any) -> List[Dict[str, Any]]:
    if pd.isna(poly_val) or not str(poly_val).strip() or str(poly_val).strip() == '[]':
        return []
    val_str = str(poly_val).strip()
    try:
        return json.loads(val_str)
    except Exception:
        pass
    try:
        return ast.literal_eval(val_str)
    except Exception:
        return []

parsed_records = []
alignment_errors = []
empty_records = []
label_counts = {}

for idx, row in df_raw.iterrows():
    img_id = str(row['img_id']).strip()
    anno_num = int(row['anno_num']) if not pd.isna(row['anno_num']) else 0
    quality = float(row['anno_image_quality']) if not pd.isna(row['anno_image_quality']) else 0.0
    
    if anno_num == 0:
        empty_records.append(img_id)
        parsed_records.append({'img_id': img_id, 'anno_num': 0, 'image_quality': quality, 'instances': []})
        continue
        
    raw_texts = str(row['anno_texts']) if not pd.isna(row['anno_texts']) else ''
    raw_labels = str(row['anno_labels']) if not pd.isna(row['anno_labels']) else ''
    
    texts = raw_texts.split('|||') if raw_texts else []
    labels = raw_labels.split('|||') if raw_labels else []
    polygons = parse_polygons_field(row['anno_polygons'])
    
    if not (len(texts) == len(labels) == len(polygons)):
        alignment_errors.append({'img_id': img_id, 'n_texts': len(texts), 'n_labels': len(labels), 'n_polygons': len(polygons)})
        continue
        
    instances = []
    for t, l, p in zip(texts, labels, polygons):
        clean_lbl = LABEL_NORMALIZATION.get(l.strip(), l.strip())
        label_counts[clean_lbl] = label_counts.get(clean_lbl, 0) + 1
        
        bbox = p.get('bbox', [0, 0, 0, 0])
        w_img = max(1, p.get('width', 1000))
        h_img = max(1, p.get('height', 1000))
        
        # Normalize bbox to [0, 1000] integer coordinates for LayoutXLM
        x0 = max(0, min(1000, int(round((bbox[0] / w_img) * 1000))))
        y0 = max(0, min(1000, int(round((bbox[1] / h_img) * 1000))))
        x1 = max(0, min(1000, int(round(((bbox[0] + bbox[2]) / w_img) * 1000))))
        y1 = max(0, min(1000, int(round(((bbox[1] + bbox[3]) / h_img) * 1000))))
        
        instances.append({
            'text': t.strip(),
            'label': clean_lbl,
            'category_id': p.get('category_id'),
            'raw_bbox': bbox,
            'norm_bbox': [min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1)],
            'segmentation': p.get('segmentation', []),
            'img_width': w_img,
            'img_height': h_img
        })
        
    parsed_records.append({'img_id': img_id, 'anno_num': len(instances), 'image_quality': quality, 'instances': instances})

print(f'Total records parsed: {len(parsed_records)}')
print(f'Valid records: {len(parsed_records) - len(alignment_errors)}')
print(f'Empty records (anno_num == 0): {len(empty_records)} -> {empty_records}')
print(f'Alignment errors: {len(alignment_errors)}')
print(f'Label counts: {label_counts}')

In [ ]:
# Check image presence on disk
disk_images = {p.name: p for p in TRAIN_IMG_DIR.glob('*.jpg')}
csv_ids = [r['img_id'] for r in parsed_records]

missing = [i for i in csv_ids if i not in disk_images]
orphaned = [name for name in disk_images if name not in set(csv_ids)]

print(f'Images referenced in CSV: {len(csv_ids)}')
print(f'Images present on disk: {len(disk_images)}')
print(f'Missing images: {len(missing)}')
print(f'Orphaned images on disk: {len(orphaned)}')

In [ ]:
# Sample resolution check
sample_ids = csv_ids[:min(200, len(csv_ids))]
widths, heights, aspect_ratios = [], [], []

for img_id in sample_ids:
    p = disk_images.get(img_id)
    if p and p.exists():
        with Image.open(p) as img:
            w, h = img.size
            widths.append(w)
            heights.append(h)
            aspect_ratios.append(h / w)

print(f'Resolution statistics (sample size = {len(widths)}):')
print(f'  Widths:  Min={min(widths)}px, Max={max(widths)}px, Median={int(np.median(widths))}px')
print(f'  Heights: Min={min(heights)}px, Max={max(heights)}px, Median={int(np.median(heights))}px')
print(f'  Aspect Ratios (H/W): Min={min(aspect_ratios):.2f}, Max={max(aspect_ratios):.2f}, Median={np.median(aspect_ratios):.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Entity counts bar chart
s_labels = pd.Series(label_counts).sort_values(ascending=False)
bars = axes[0].bar(s_labels.index, s_labels.values, color=['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728'])
axes[0].set_title('Target Entity Frequency in Ground Truth', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Annotation Count')
for bar in bars:
    y = bar.get_height()
    axes[0].text(bar.get_x() + bar.get_width() / 2.0, y + 25, f'{y:,}', ha='center', va='bottom', fontsize=10)

# Quality score histogram
qualities = [r['image_quality'] for r in parsed_records]
axes[1].hist(qualities, bins=25, color='#4c72b0', edgecolor='white')
axes[1].axvline(float(np.mean(qualities)), color='red', linestyle='--', label=f'Mean: {np.mean(qualities):.2f}')
axes[1].set_title('Receipt Image Quality Score (Task 1)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Quality Score (0.0 = degraded, 1.0 = crisp)')
axes[1].set_ylabel('Invoice Count')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
COLOR_MAP = {
    'SELLER': '#2ca02c',
    'ADDRESS': '#1f77b4',
    'TIMESTAMP': '#ff7f0e',
    'TOTAL_COST': '#d62728'
}

def visualize_sample(record: Dict[str, Any]):
    img_id = record['img_id']
    p = disk_images.get(img_id)
    if not p:
        return
    img = Image.open(p).convert('RGB')
    fig, ax = plt.subplots(figsize=(8, 11))
    ax.imshow(img)
    ax.set_title(f'Receipt: {img_id} | Annos: {record["anno_num"]} | Quality: {record["image_quality"]:.2f}', fontsize=11, fontweight='bold')
    for inst in record['instances']:
        b = inst['raw_bbox']
        lbl = inst['label']
        col = COLOR_MAP.get(lbl, 'purple')
        rect = patches.Rectangle((b[0], b[1]), b[2], b[3], linewidth=2, edgecolor=col, facecolor='none')
        ax.add_patch(rect)
        ax.text(b[0], max(0, b[1] - 5), f'{lbl}: {inst["text"][:20]}',
                bbox=dict(boxstyle='square,pad=0.2', facecolor=col, alpha=0.8, edgecolor='none'),
                fontsize=8, color='white', fontweight='bold')
    ax.axis('off')
    plt.tight_layout()
    plt.show()

non_empty = [r for r in parsed_records if r['anno_num'] > 0]
for sample in non_empty[:2]:
    visualize_sample(sample)

In [ ]:
out_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('reports')
out_dir.mkdir(parents=True, exist_ok=True)
report_path = out_dir / 'mc_ocr_data_quality_report.json'

report_data = {
    'dataset_name': 'MC-OCR 2021',
    'total_records': len(parsed_records),
    'valid_records': len(parsed_records) - len(alignment_errors),
    'alignment_errors': len(alignment_errors),
    'empty_records_count': len(empty_records),
    'empty_record_ids': empty_records,
    'total_images_on_disk': len(disk_images),
    'label_distribution': label_counts,
    'quality_score_mean': float(np.mean(qualities)),
    'quality_score_std': float(np.std(qualities)),
    'dimensions_summary': {
        'median_width': int(np.median(widths)) if widths else None,
        'median_height': int(np.median(heights)) if heights else None,
        'median_aspect_ratio': float(np.median(aspect_ratios)) if aspect_ratios else None
    }
}

with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report_data, f, indent=2, ensure_ascii=False)

print(f'==> Data quality report successfully exported to: {report_path}')
print(json.dumps(report_data, indent=2, ensure_ascii=False))